# Final Capstone: MLB & Pitch Clock Analytics 

In [1]:
# import all requirements
import pandas as pd 
import requests 
import sqlalchemy
import notebook
from sqlalchemy import create_engine
from sqlalchemy import text

---
## Part 1: Data Ingestion

In this part you will load both data sources into pandas DataFrames. Nothing gets cleaned yet — just get the raw data in.

In [2]:

def get_all_player_ids(season=2025):
    
    teams_url = "https://statsapi.mlb.com/api/v1/teams"
    params = {"sportId": 1, "season": season} 
    
    response = requests.get(teams_url, params=params)
    response.raise_for_status()
    teams_data = response.json()
    
    all_players = {}
    
    for team in teams_data.get("teams", []):
        team_id = team["id"]
        team_name = team["name"]
        
        roster_url = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/roster"
        roster_params = {"season": season}
        
        roster_response = requests.get(roster_url, params=roster_params)
        roster_response.raise_for_status()
        roster_data = roster_response.json()
        
        for player in roster_data.get("roster", []):
            player_id = player["person"]["id"]
            player_name = player["person"]["fullName"]
            
            if player_id not in all_players:
                all_players[player_id] = {
                    "name": player_name,
                    "team": team_name,
                    "team_id": team_id,
                    "position": player.get("position", {}).get("abbreviation", "N/A")
                }
    
    return all_players

players = get_all_player_ids(season=2025)
print(f"Found {len(players)} active players")

df_players = pd.DataFrame([
    {"player_id": player_id, **player_info} 
    for player_id, player_info in players.items()
])

print(df_players.head())

all_player_ids = df_players['player_id'].tolist()
print(f"\nAll player IDs (first 10): {all_player_ids[:10]}")

Found 1470 active players
   player_id             name       team  team_id position
0     680769     CJ Alexander  Athletics      133       1B
1     669920  Jason Alexander  Athletics      133        P
2     665660   Elvis Alvarado  Athletics      133        P
3     609280   Miguel Andujar  Athletics      133       LF
4     682183       Drew Avans  Athletics      133       LF

All player IDs (first 10): [680769, 669920, 665660, 609280, 682183, 686930, 669620, 674370, 668709, 641386]


In [6]:

def fetch_mlb_api(endpoint, **params):
    base_url = "https://statsapi.mlb.com/api/v1"
    url = f"{base_url}/{endpoint}"
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

def extract_player_stats_to_dataframe(api_response):
    if 'stats' in api_response and api_response['stats']:
        stats_data = api_response['stats'][0]
        
        if 'splits' in stats_data and stats_data['splits']:
            player_stats = stats_data['splits'][0]
            
            stat_dict = player_stats.get('stat', {})
            metadata = {
                'player_id': player_stats.get('player', {}).get('id'),
                'player_name': player_stats.get('player', {}).get('fullName'),
                'season': player_stats.get('season'),
                'team_id': player_stats.get('team', {}).get('id'),
                'team_name': player_stats.get('team', {}).get('name'),
                'league_id': player_stats.get('league', {}).get('id'),
                'league_name': player_stats.get('league', {}).get('name'),
                'game_type': player_stats.get('gameType'),
                'stat_type': stats_data.get('type', {}).get('displayName'),
                'stat_group': stats_data.get('group', {}).get('displayName')
            }
            
            combined_dict = {**metadata, **stat_dict}
            
            for key, value in combined_dict.items():
                if isinstance(value, str) and value.replace('.', '').replace('-', '').isdigit():
                    if '.' in value:
                        combined_dict[key] = float(value)
                    else:
                        combined_dict[key] = int(value)
            
            df = pd.DataFrame([combined_dict])
            
            return df
    
    return pd.DataFrame()

all_data = []

for player in all_player_ids:
    endpoint = f"people/{player}/stats"
    params = {
        "stats": "season",
        "season": 2025,
        "gameType": "R" 
    }
    player = fetch_mlb_api(endpoint, **params)
    player = extract_player_stats_to_dataframe(player)
    all_data.append(player)

mlb_data = pd.concat(all_data, ignore_index=True)
print(mlb_data)

      player_id      player_name  season team_id      team_name league_id  \
0        680769     CJ Alexander    2025     133      Athletics       103   
1        669920  Jason Alexander    2025    None           None       103   
2        665660   Elvis Alvarado    2025     133      Athletics       103   
3        609280   Miguel Andujar    2025    None           None      None   
4        682183       Drew Avans    2025    None           None      None   
...         ...              ...     ...     ...            ...       ...   
1464     663399  Brandon Waddell    2025     121  New York Mets       104   
1465     681810    Austin Warren    2025     121  New York Mets       104   
1466     608385     Jesse Winker    2025     121  New York Mets       104   
1467     664849      Danny Young    2025     121  New York Mets       104   
1468     676724      Jared Young    2025     121  New York Mets       104   

          league_name game_type stat_type stat_group  ...  pitchesPerInning

---
## Part 2: Exploratory Data Analysis (EDA)

Before cleaning anything, understand what you have. For each table check:
- **Shape** — rows and columns
- **Dtypes** — are dates stored as strings? Numbers as objects?
- **Nulls** — which columns have missing values?
- **Duplicates** — any exact duplicate rows?
- **Value distributions** — unique values in key categorical columns

Useful methods: `.shape`, `.dtypes`, `.isnull().sum()`, `.duplicated().sum()`, `.value_counts()`, `.describe()`

In [8]:
print(mlb_data.shape)
print(mlb_data.dtypes)
mlb_data.head(5)


NameError: name 'mlb_data' is not defined

In [ ]:
# How many null values?

for columns in mlb_data:
    print(mlb_data.isnull().sum())

In [ ]:
# How many duplicate rows are there?

# YOUR CODE HERE
print(launches.duplicated().sum())

# What are the unique values (and counts) in the 'agency' column?

# YOUR CODE HERE
print(launches['agency'].unique())

print(launches.value_counts('agency'))

# What are the unique values (and counts) in the 'outcome' column?

# YOUR CODE HERE
print(launches['outcome'].unique())

print(launches.value_counts('outcome'))

# How many launches per destination?

# YOUR CODE HERE
launches_per_destination = launches['destination'].value_counts().sort_index() 
print(launches['destination'].value_counts().sort_index())

In [ ]:
# Explore the MLB DataFrame

print(mlb_data.shape)
print(mlb_data.dtypes)
print(mlb_data.value_counts())
print(mlb_data.duplicated().sum())
print(mlb_data.isnull().sum())
print(mlb_data.describe())


---
## Part 3: Data Cleaning

Fix every issue you found in EDA. Only after cleaning should data be written to the database.